In [2]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, MessagesState,START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
load_dotenv()

model =ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.environ["GROQ_API_KEY"],
    streaming=True
)

def chatbot(state:MessagesState):
    response =model.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot",chatbot)
builder.add_edge(START,"chatbot")
builder.add_edge("chatbot", END)
checkpointer = InMemorySaver()

app = builder.compile(checkpointer=checkpointer)

config ={
    "configurable":{"thread_id":"chat-1"}
}

res0 = app.invoke(
    {"messages":[HumanMessage(content="My name is mayank")]},
    config
)

result = app.invoke(
    {"messages":[HumanMessage(content="what is my name")]},
    config
)

print(result["messages"][-1].content)


Your name is Mayank.
